# **Hate Speech Detection**: Logistic regression 

_Importing functionality...._

In [1]:
# Importing sklearn functionality
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

# Importing data science libraries
import numpy as np
import pandas as pd

# Importing custom pre-process text
import sys
import os
from dotenv import load_dotenv
from pathlib import Path

_Importing the pre-process functionality..._

In [2]:
# Add the 'your_project' root directory to sys.path
# This assumes your_project is the parent of both 'data' and 'utils'
# os.path.abspath(__file__) would give you the notebook's path,
# then two '..' go up to 'your_project'
module_path = os.path.abspath(os.path.join('..', os.path.dirname('')))
if module_path not in sys.path:
    sys.path.append(module_path)


# Importing NLTK functionality
from utils.nlp import preprocess_for_basic_nlp

# Importing database connection
from utils.database import Database

_Connecting to the database..._

In [3]:
# Ensrung we're at the root of the project
project_root = Path(os.getcwd()).resolve().parents[0]

# Constructin the path of the environment variable 
dotenv_path = project_root / 'credentials.env'

# Connecting to the correct environment path
load_dotenv(dotenv_path = dotenv_path)

# Retrieving properties of username
db_password = os.getenv('DB_PASSWORD')
db_string   = os.getenv('DB_STRING')

# Putting the password in the string
db_string = db_string.replace('<db_password>', db_password)

# Name of my database
db_name = 'Hate_App'
    
# Creating the database object
db = Database(connection_string = db_string, db_name = db_name)

# Name of collection to write
col_write = 'text'

_Retrieving the data..._

In [4]:
# Query for English
query_english = {'language' : {'$eq' : 'English'}}

# Retrieving the data
eng_dict = db.get_records(collection_name = col_write, query = query_english)

In [6]:
# Converting into a dataframe
df = pd.DataFrame(eng_dict)
df.head(3)

,_id,text,processed_text,language,dataset,label
0,68817d2597a83e535527e8a7,"Even dah setiap hari tengok pon, still terseny...","Even dah setiap hari tengok pon, still terseny...",English,ALD,0
1,68817d2597a83e535527e8ec,USER Ah anjir favorit gue bgt',USER Ah anjir favorit gue bgt',English,ALD,1
2,68817d2597a83e535527e921,USER bacot bat bgst',USER bacot bat bgst',English,ALD,1


_Selecting the two features..._

In [7]:
# Label is the target
y = df['label'].to_list()

# Selecting the featues
X = df['text'].to_list()

# Processing the text
X_processed = [preprocess_for_basic_nlp(x) for x in X]

_Splitting the dataset.._

In [8]:
# Splitting in to train / test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

_Creating the features..._

In [9]:
# Creatin the TF-IDF vectorizer
tfidf = TfidfVectorizer()

# Transformating the train
X_train_feat = tfidf.fit_transform(X_train)

# Transofmring the test
X_test_feat  = tfidf.transform(X_test)

_Fitting a logistic regression..._

In [10]:
# Creating the logistic regression
lr = LogisticRegression()

# Fitting the model
lr.fit(X_train_feat, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


_Evaluating the model..._

In [11]:
# Making training predictions
y_train_pred = lr.predict(X_train_feat)

# Producing the classification report
print(classification_report(y_true = y_train, y_pred = y_train_pred))

              precision    recall  f1-score   support

           0       0.75      0.67      0.71     16136
           1       0.76      0.83      0.79     20640

    accuracy                           0.76     36776
   macro avg       0.76      0.75      0.75     36776
weighted avg       0.76      0.76      0.76     36776



_Checking on test..._

In [12]:
# Making a prediction of test
y_test_pred = lr.predict(X_test_feat)

# Creating the test classification report
print(classification_report(y_true = y_test, y_pred = y_test_pred))

              precision    recall  f1-score   support

           0       0.63      0.53      0.57      4108
           1       0.66      0.75      0.70      5087

    accuracy                           0.65      9195
   macro avg       0.65      0.64      0.64      9195
weighted avg       0.65      0.65      0.65      9195



_Saving the models.._

In [13]:
# Importing joblib
import joblib

# Path of models data
path_output = 'LR_English_TFIDF_TM_20250629.joblib'

# Saving te result
joblib.dump(
    {
        'model': lr,
        'vectorizer': tfidf
    }, 
    path_output
)

['LR_English_TFIDF_TM_20250629.joblib']

#